In [1]:
# 1. Basic imports
import pandas as pd
import numpy as np

# NLP and modeling
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

# Optional: for nicer text preprocessing
import re

In [3]:
# Path to your CSV
csv_path = "barbour_sales_uk.csv"   # change as needed

df = pd.read_csv(csv_path)

# quick peek
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   OrderID       15 non-null     int64  
 1   Date          15 non-null     object 
 2   Store         15 non-null     object 
 3   City          15 non-null     object 
 4   Product       15 non-null     object 
 5   Category      15 non-null     object 
 6   Quantity      15 non-null     int64  
 7   UnitPrice     15 non-null     float64
 8   TotalPrice    15 non-null     float64
 9   CustomerType  15 non-null     object 
dtypes: float64(2), int64(2), object(6)
memory usage: 1.3+ KB


In [5]:
df.columns = df.columns.str.strip().str.lower()

In [7]:
df.head()
df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   orderid       15 non-null     int64  
 1   date          15 non-null     object 
 2   store         15 non-null     object 
 3   city          15 non-null     object 
 4   product       15 non-null     object 
 5   category      15 non-null     object 
 6   quantity      15 non-null     int64  
 7   unitprice     15 non-null     float64
 8   totalprice    15 non-null     float64
 9   customertype  15 non-null     object 
dtypes: float64(2), int64(2), object(6)
memory usage: 1.3+ KB


In [9]:
if "price" in df.columns and "cost" in df.columns:
    df = df.dropna(subset=["price", "cost"])
else:
    print("Columns not found:", df.columns)

Columns not found: Index(['orderid', 'date', 'store', 'city', 'product', 'category', 'quantity',
       'unitprice', 'totalprice', 'customertype'],
      dtype='object')


In [10]:
numeric_cols = []
for c in ["price", "cost", "profit", "quantity"]:
    if c in df.columns:
        numeric_cols.append(c)

In [11]:
text_col = "customer_notes"   # change to your free-text column

if text_col in df.columns:
    # basic cleaning function
    def clean_text(s):
        if pd.isna(s):
            return ""
        # lower, remove non-alphanum except spaces
        return re.sub(r"[^a-z0-9 ]", " ", str(s).lower())

    df["text_clean"] = df[text_col].apply(clean_text)

    # TF-IDF vectorization
    tfidf = TfidfVectorizer(
        max_features=500,   # adjust based on dataset size
        stop_words="english",
        ngram_range=(1,2)
    )
    X_text = tfidf.fit_transform(df["text_clean"])
else:
    X_text = None

In [12]:
from scipy import sparse

# numeric matrix
if numeric_cols:
    X_numeric = df[numeric_cols].fillna(0).values
else:
    X_numeric = np.zeros((len(df),0))

# combine
if X_text is not None:
    X = sparse.hstack([X_numeric, X_text])
else:
    X = X_numeric

In [15]:
print(X.shape)

(15, 1)


In [22]:
print(list(df.columns))

['orderid', 'date', 'store', 'city', 'product', 'category', 'quantity', 'unitprice', 'totalprice', 'customertype']


In [26]:
df.columns

Index(['orderid', 'date', 'store', 'city', 'product', 'category', 'quantity',
       'unitprice', 'totalprice', 'customertype'],
      dtype='object')